# Qorğan — fine-tune XLM-R scam classifier (D3-1)

Fine-tunes `xlm-roberta-base` with a **binary risk head** + **multi-label tactic head** on the
Qorğan corpus, fits a calibration temperature on the val split, and exports a self-describing
bundle that the `xlmr` backend (`qorgan.classifier.predict`) loads.

Runs on a free Colab **T4 GPU** (Runtime → Change runtime type → GPU) or locally on Apple **MPS**.
All logic lives in `qorgan.classifier.train` — this notebook is just the driver so the training
path is identical to `python -m qorgan.classifier.train`.

## 1. Setup (Colab only — skip locally if the repo is already installed)

In [ ]:
# On Colab: clone the repo and install it (skip if running from the repo locally).
# !git clone https://github.com/<your-org>/govtech.git
# %cd govtech
# !pip install -q -e .
#
# You also need the processed corpus (data/processed/{train,val}.jsonl). Either build it
# (needs GEMINI_API_KEY) or upload the JSONL splits:
# !python -m qorgan.data.build_corpus --config configs/corpus.yaml

## 2. Fine-tune + calibrate + export

In [ ]:
from pathlib import Path

from transformers import AutoTokenizer

from qorgan.classifier import labels
from qorgan.classifier.model import build_model
from qorgan.classifier.train import train_and_export, pick_device
from qorgan.config import get_config
from qorgan.eval.run import load_split

cfg = get_config()
BASE_MODEL = "xlm-roberta-base"
processed = cfg.data_dir / "processed"

label_space = labels.default_label_space()
print("device:", pick_device(), "| tactics:", len(label_space))

model = build_model(BASE_MODEL, num_tactics=len(label_space))
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

metadata = train_and_export(
    load_split(processed, "train"),
    load_split(processed, "val"),
    model=model,
    tokenizer=tokenizer,
    base_model=BASE_MODEL,
    label_space=label_space,
    out_dir=cfg.xlmr_model_dir,
    epochs=3,
    batch_size=8,
    lr=2e-5,
    max_length=256,
)
metadata

## 3. FPR-first evaluation of the trained model (test + real_heldout, separately)

In [ ]:
from qorgan.eval.run import run, format_report

results = run(processed, ["test", "real_heldout"], backend="xlmr")
print(format_report(results))

## 4. Download the exported bundle (Colab)
The `models/xlmr/` dir (`model.pt` + `metadata.json`) is what the demo app / eval load via the
`xlmr` backend. On Colab, zip and download it; locally it is already in place.

In [ ]:
# Colab download:
# import shutil; from google.colab import files
# shutil.make_archive('xlmr_model', 'zip', cfg.xlmr_model_dir)
# files.download('xlmr_model.zip')